# C01 — Leitura e tradução do corpus

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Este notebook realiza a primeira etapa do projeto da disciplina **"Sistemas Cognitivos com LLMs"**, responsável por
carregar as transcrições de aula em português disponíveis em `data/raw`, traduzi-las
automaticamente para espanhol e salvar os dados processados para as próximas etapas do
pipeline, a partir de C02.

O objetivo é permitir que o sistema processe as transcrições em outro idioma, mantendo a
fidelidade ao conteúdo original e a rastreabilidade por arquivo e linha, ou cue.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🎯 Objetivos desta etapa**

- Localizar automaticamente os documentos `.vtt`/`.txt` em português na pasta `data/raw`.
- Limpar as marcas de tempo e índices dos arquivos WEBVTT, mantendo uma linha por cue.
- Traduzir o conteúdo de português para espanhol com **três métodos diferentes**, para
  comparar qualidade e custo: dois modelos locais dedicados, `facebook/nllb-200-distilled-600M`
  e `Helsinki-NLP/opus-mt-tc-big-itc-itc`, e a API remota do **Gemini 3.1 Pro (Preview)**.
- Garantir determinismo e reprodutibilidade da tradução entre execuções.
- Salvar a tradução do NLLB, usada nas próximas etapas do pipeline, em
  `<nome-original>_espanhol.txt`, e o texto português já limpo, sem timestamps nem índices
  mas sem traduzir, em `<nome-original>_portugues.txt`, ambos na pasta `data/processed/`.
- Preparar o corpus traduzido, e seu par em português, para as próximas etapas do pipeline:
  comparação bilíngue de técnicas de prompting em C02 e, futuramente, embeddings, busca
  vetorial e RAG em C05.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🌐 Convenções deste notebook**

- **Três métodos de tradução comparados**: dois modelos locais — `facebook/nllb-200-distilled-600M` (Meta) e `Helsinki-NLP/opus-mt-tc-big-itc-itc` (Univ. de Helsinki) — e a API do **Gemini 3.1 Pro (Preview)**. O NLLB gera os arquivos usados nas próximas etapas; os outros dois entram só para comparar qualidade e custo.
- **Limpeza WEBVTT**: os `.vtt` são limpos de marcas de tempo e índices antes de traduzir, mantendo **uma linha por cue**, de modo que a tradução preserva as mesmas quebras de linha do original.
- **Determinismo e reprodutibilidade** da tradução entre execuções.
- **Variáveis, funções e descrições em português**.
- **Saídas** em `data/processed/`: `<nome>_espanhol.txt` (tradução do NLLB, usada adiante) e `<nome>_portugues.txt` (português limpo, sem traduzir).

**Entrada**: lê os `.vtt`/`.txt` em português de `data/raw/` — por ser a primeira etapa do pipeline, não depende de nenhum notebook anterior.

</div>

In [1]:
%pip install transformers torch sentencepiece python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
login(token=os.getenv("HUGGINGFACE_KEY"))

---

## §1 Primeiro modelo local: NLLB-200-distilled-600M

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Por que escolhemos o NLLB-200-distilled-600M**

A tarefa de traduzir exige compreensão completa da entrada, algo que um decoder-only não faz
tão bem, e geração de uma sequência nova, algo que um encoder-only, como BERT ou DistilBERT,
não consegue fazer de forma alguma — já buscávamos, portanto, um Transformer
**encoder-decoder**. Escolhemos especificamente o `nllb-200-distilled-600M`, da Meta, por
motivos concretos:

- **Cobertura multilíngue real**: faz parte da família NLLB-200, treinada para 200 idiomas,
  incluindo exatamente os dois que precisamos, `por_Latn` e `spa_Latn` — não é um modelo
  genérico que "também" traduz, é feito para tradução entre muitos pares de idiomas.
- **Aberto e gratuito**: é um checkpoint público no Hugging Face, sem custo de licença nem de
  API, o que importa porque comparamos justamente contra a alternativa paga, o Gemini, da
  seção da API do Gemini até a seção de conclusão.
- **Versão destilada, do tamanho certo pra tarefa**: a Meta treinou primeiro um modelo bem
  maior, com bilhões de parâmetros, cobrindo 200 idiomas, e depois treinou esta versão de
  600 milhões de parâmetros para imitar as respostas do modelo grande, com uma fração do
  custo computacional. Em vez de rodar o NLLB-200 gigante, usamos a versão destilada, boa o
  suficiente para tradução entre duas línguas próximas, português e espanhol.

**✅ Arquitetura e como aparece no código**

`nllb-200-distilled-600M` é um Transformer **encoder-decoder**:

| Componente | Papel | Atenção |
|---|---|---|
| 🔎 Encoder | Entende a linha de origem completa antes de traduzir | Bidirecional |
| ✍️ Decoder | Gera a frase em espanhol token a token | Autorregressiva, com cross-attention sobre o encoder |

Não é só uma escolha teórica — é assim que os dois papéis aparecem no código:

- **Classe do `transformers`**: `AutoModelForSeq2SeqLM` para o NLLB, célula `setup`; o
  Helsinki-NLP usa `MarianMTModel`, célula `b29bd053`, discutido na seção do Helsinki-NLP —
  as duas já vêm com a arquitetura encoder-decoder pronta, não escolhemos isso manualmente.
- **Encoder**: `tok_tradutor(linhas, ...)` transforma o lote de linhas em português em
  tokens, e é esse resultado que entra no encoder dentro de `model_tradutor.generate(...)` —
  o encoder já processou a frase inteira antes de o decoder escrever a primeira palavra.
- **Decoder**: a mesma chamada `generate(...)` roda o decoder token a token, de forma
  autorregressiva, usando cross-attention sobre o que o encoder entendeu. Como o decoder
  precisa saber em qual idioma escrever, o primeiro token que ele gera é fixado de fora,
  `forced_bos_token_id` no NLLB — detalhado na seção da tokenização.

</div>

In [3]:
import re
from pathlib import Path
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

arquivos_antigos = [f for f in PROCESSED_DIR.iterdir() if f.is_file()]
for arquivo_antigo in arquivos_antigos:
    arquivo_antigo.unlink()
print(f"{len(arquivos_antigos)} arquivo(s) antigo(s) removido(s) de {PROCESSED_DIR} antes de gerar os novos")

MODEL_ID = "facebook/nllb-200-distilled-600M"

if torch.cuda.is_available():
    DEVICE, DEVICE_NOME = "cuda", "cuda (GPU NVIDIA ou AMD via ROCm)"
elif torch.backends.mps.is_available():
    DEVICE, DEVICE_NOME = "mps", "mps (GPU Apple Silicon)"
else:
    DEVICE, DEVICE_NOME = "cpu", "cpu"
    try:
        import torch_directml
        if torch_directml.is_available():
            DEVICE = torch_directml.device()
            DEVICE_NOME = "directml (GPU AMD/Intel/NVIDIA via DirectX 12, Windows/WSL)"
    except Exception:
        pass

print(f"Dispositivo de inferência: {DEVICE_NOME}")

tok_tradutor = AutoTokenizer.from_pretrained(MODEL_ID)
model_tradutor = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

tok_tradutor.src_lang = "por_Latn"
FORCED_BOS_TOKEN_ID = tok_tradutor.convert_tokens_to_ids("spa_Latn")

# Parâmetros de geração compartilhados entre NLLB e Helsinki-NLP — mesma justificativa de
# determinismo/beam-search para os dois (ver a seção dos parâmetros de geração e a seção do
# Helsinki-NLP), declarados uma única vez para os dois modelos não divergirem por engano.
PARAMETROS_GERACAO = dict(
    num_beams=4,
    do_sample=False,
    early_stopping=True,
    length_penalty=1.0,
    no_repeat_ngram_size=3,
)

def traduzir_lote(linhas: list[str], max_length: int = 512) -> list[str]:
    entrada = tok_tradutor(
        linhas,
        return_tensors="pt", padding=True, truncation=True, max_length=max_length
    ).to(DEVICE)
    saida = model_tradutor.generate(
        **entrada,
        forced_bos_token_id=FORCED_BOS_TOKEN_ID,
        max_length=max_length,
        **PARAMETROS_GERACAO,
    )
    return [tok_tradutor.decode(s, skip_special_tokens=True) for s in saida]

0 arquivo(s) antigo(s) removido(s) de data/processed antes de gerar os novos
Dispositivo de inferência: cuda (GPU NVIDIA ou AMD via ROCm)


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

---

## §2 Justificativa da tokenização

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Justificativa da tokenização**

O tokenizer do NLLB usa **SentencePiece**, uma tokenização de subpalavras, com um
vocabulário compartilhado entre **200 idiomas** de famílias bem diferentes:

1. **Vocabulário multilíngue amplo**: por cobrir 200 idiomas com um único checkpoint, o
   vocabulário é grande e genérico. Não é afinado especificamente para o par
   português-espanhol, mas é suficiente para uma tradução entre duas línguas próximas.
2. **Idioma de destino como parâmetro, não como texto**: fixamos
   `tok_tradutor.src_lang = "por_Latn"`, o idioma de origem, e passamos o idioma de destino
   separadamente, como `forced_bos_token_id` no método `generate`. O primeiro token que o decoder
   é obrigado a gerar é o código do idioma de destino, `spa_Latn`. Isso mantém a indicação de
   idioma fora do texto de entrada, sem qualquer ambiguidade entre o que é instrução e o que
   é conteúdo a traduzir.

`padding=True` e `truncation=True` são necessários porque cada lote, de tamanho
`BATCH_SIZE=16`, agrupa linhas de comprimento variável, e o tensor de entrada precisa que
todas tenham o mesmo tamanho; `max_length=512` respeita o limite de posições do modelo.

</div>

---

## §3 Parâmetros de geração

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Parâmetros de geração**

Diferente de tarefas de geração criativa de texto, onde se usa `do_sample=True` mais
`temperature` para variar as respostas, aqui se busca **fidelidade e reprodutibilidade**,
não variedade: uma tradução não deveria mudar cada vez que o notebook é executado. Por isso:

- **`num_beams=4`**: faz *beam search* em vez de *greedy decoding*, que só olha o token mais
  provável em cada passo. Explora 4 sequências candidatas em paralelo e retorna a de maior
  probabilidade conjunta ao final. Deixamos explícito aqui para que fique documentado no
  código, em vez de depender de um default implícito da biblioteca.
- **`do_sample=False`**: decodificação determinística. Para tradução não queremos
  aleatoriedade — a mesma linha de entrada deve sempre produzir a mesma tradução.
- **`early_stopping=True`**: interrompe a busca assim que os 4 feixes já geraram o token de
  fim de sequência, evitando computação extra desnecessária.
- **`length_penalty=1.0`**: neutro, nem penaliza nem premia traduções mais longas. Sendo
  português-espanhol um par de idiomas com comprimentos de frase muito similares, não há
  razão para enviesar o tamanho da saída.
- **`no_repeat_ngram_size=3`**: proíbe repetir qualquer sequência de 3 tokens já gerada.
  Mitiga os loops de repetição que esses modelos de tradução pequenos ocasionalmente
  produzem, um artefato de decodificação diferente dos erros de nomes próprios discutidos
  na seção que compara os três modelos.

Esses cinco valores vivem numa única constante, `PARAMETROS_GERACAO`, definida uma vez nesta
célula e reaproveitada tanto por `traduzir_lote` (NLLB) quanto por `traduzir_lote_helsinki`
(Helsinki-NLP, seção do Helsinki-NLP) — assim os dois modelos usam de fato os mesmos
parâmetros, garantido pelo código, não só reafirmado por descrição em cada seção.

**✅ Uso de memória da GPU**: `num_beams=4` faz o *beam search* processar, na prática, 4
vezes mais sequências ao mesmo tempo do que o tamanho do lote sugere — a memória usada
cresce com `BATCH_SIZE` vezes `num_beams`. Duas escolhas cuidam disso:

- **Carregamos o modelo em `float16`**, metade da memória de `float32`, quando há GPU
  disponível. Perdemos um pouco de precisão numérica, mas não o suficiente para afetar a
  qualidade da tradução, só o suficiente para caber confortavelmente na memória da placa de
  vídeo.
- **`BATCH_SIZE=16`**: esse número já foi testado nesta GPU e funcionou; um valor maior,
  como 32, chegou a estourar a memória, erro `CUDA error: out of memory`, durante o *beam
  search*. Se sua GPU tiver mais memória disponível, pode aumentar esse valor; se ainda
  assim faltar memória, reduza mais.

</div>

In [4]:
def eh_webvtt(texto: str) -> bool:
    return texto.lstrip().startswith("WEBVTT")


def extrair_linhas(texto: str) -> list[str]:
    if not eh_webvtt(texto):
        return [l.strip() for l in texto.splitlines() if l.strip()]

    return [
        l.strip() for l in texto.splitlines()
        if l.strip()
        and not l.startswith("WEBVTT")
        and not re.match(r'^\d+$', l.strip())
        and not re.match(r'^\d{2}:\d{2}', l.strip())
    ]

In [5]:
from IPython.display import display, HTML

documentos_pt = [
    f for f in sorted(RAW_DIR.glob("*"))
    if f.suffix.lower() in (".txt", ".vtt") and not f.stem.endswith("_espanhol")
]

_itens = "".join(f"<li><code>{f.name}</code></li>" for f in documentos_pt)
display(HTML(
    '<div style="background:#eef6ff;border-left:4px solid #2b7de9;border-radius:4px;'
    'padding:10px 14px;margin:6px 0;color:#111827;font-family:system-ui,-apple-system,sans-serif;">'
    f'<strong>📄 {len(documentos_pt)} documento(s) em português encontrado(s)</strong> '
    f'em <code>{RAW_DIR}</code>'
    f'<ul style="margin:6px 0 0 0;padding-left:20px;line-height:1.6;">{_itens}</ul>'
    '</div>'
))

In [6]:
from IPython.display import display, HTML

BATCH_SIZE = 16

_resultados = []
for arquivo in documentos_pt:
    texto_original = arquivo.read_text(encoding="utf-8")
    linhas = extrair_linhas(texto_original)

    # Guarda o texto português limpo (antes de traduzir) para uso bilíngue (ES/PT) em c02.
    arquivo_saida_pt = PROCESSED_DIR / f"{arquivo.stem}_portugues.txt"
    arquivo_saida_pt.write_text("\n".join(linhas), encoding="utf-8")

    linhas_traduzidas = []
    for i in range(0, len(linhas), BATCH_SIZE):
        lote = linhas[i:i + BATCH_SIZE]
        linhas_traduzidas.extend(traduzir_lote(lote))

    texto_traduzido = "\n".join(linhas_traduzidas)

    arquivo_saida = PROCESSED_DIR / f"{arquivo.stem}_espanhol.txt"
    arquivo_saida.write_text(texto_traduzido, encoding="utf-8")

    _resultados.append({
        "arquivo": arquivo.name,
        "linhas": len(linhas),
        "caracteres": len(texto_traduzido),
        "saidas": f"{arquivo_saida.name} + {arquivo_saida_pt.name}",
    })

# Saída estilizada (mesma linguagem visual azul do restante do projeto), em vez de print().
_linhas_html = "".join(
    '<tr style="background:{bg};">'
    '<td style="padding:4px 10px;"><code>{arquivo}</code></td>'
    '<td style="padding:4px 10px;text-align:right;">{linhas}</td>'
    '<td style="padding:4px 10px;text-align:right;">{caracteres}</td>'
    '<td style="padding:4px 10px;"><code>{saidas}</code></td>'
    '</tr>'.format(bg=("#f7fbff" if n % 2 else "#ffffff"), **r)
    for n, r in enumerate(_resultados)
)
display(HTML(
    '<div style="background:#eef6ff;border-left:4px solid #2b7de9;border-radius:4px;'
    'padding:10px 14px;margin:6px 0;color:#111827;font-family:system-ui,-apple-system,sans-serif;">'
    f'<strong>✅ Tradução NLLB concluída</strong> — {len(_resultados)} aula(s), '
    f'<code>BATCH_SIZE={BATCH_SIZE}</code>, salvas em <code>{PROCESSED_DIR}</code>'
    '<table style="border-collapse:collapse;margin-top:8px;font-size:0.92em;width:100%;">'
    '<thead><tr style="background:#eef6ff;text-align:left;">'
    '<th style="padding:5px 10px;">Arquivo</th>'
    '<th style="padding:5px 10px;text-align:right;">Linhas</th>'
    '<th style="padding:5px 10px;text-align:right;">Caracteres</th>'
    '<th style="padding:5px 10px;">Arquivos gerados</th>'
    f'</tr></thead><tbody>{_linhas_html}</tbody></table>'
    '</div>'
))

Arquivo,Linhas,Caracteres,Arquivos gerados
Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08.vtt,693,84630,Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10.vtt,809,79660,Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07.vtt,794,88635,Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05.vtt,843,85008,Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-17-06-2026_20-05.vtt,835,75200,Sistemas_Cognitivos_com_LLMs_aula-17-06-2026_20-05_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-17-06-2026_20-05_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-22-06-2026_20-11.vtt,846,72655,Sistemas_Cognitivos_com_LLMs_aula-22-06-2026_20-11_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-22-06-2026_20-11_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-23-06-2026_20-08.vtt,820,70094,Sistemas_Cognitivos_com_LLMs_aula-23-06-2026_20-08_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-23-06-2026_20-08_portugues.txt
Sistemas_Cognitivos_com_LLMs_aula-25-06-2026_20-13.vtt,871,87698,Sistemas_Cognitivos_com_LLMs_aula-25-06-2026_20-13_espanhol.txt + Sistemas_Cognitivos_com_LLMs_aula-25-06-2026_20-13_portugues.txt


---

## §4 Segundo modelo local: Helsinki-NLP

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Por que escolhemos o Helsinki-NLP (`opus-mt-tc-big-itc-itc`)**

Um único modelo local testado, o NLLB-200, não é o suficiente para afirmar que nenhum
modelo local pequeno resolve os problemas de tradução que vamos ver na comparação com o
Gemini, na seção que compara os três modelos — poderia ser só uma limitação específica do
NLLB. Por isso testamos um segundo modelo, e escolhemos especificamente
`Helsinki-NLP/opus-mt-tc-big-itc-itc` por motivos concretos:

- **Arquitetura genuinamente diferente**: vem da família **OPUS-MT**, da Universidade de
  Helsinki, não é apenas "mais um checkpoint NLLB" — dá um segundo ponto de comparação real,
  para saber se a limitação observada é do modelo específico ou do tipo de abordagem, um
  modelo pequeno e especializado, sem conhecimento de mundo.
- **Cobertura de idiomas certa pra tarefa**: cobre a família de idiomas itálicos, `itc-itc`,
  que inclui exatamente português e espanhol, os dois idiomas que precisamos.
- **Aberto e gratuito**: assim como o NLLB, é um checkpoint público no Hugging Face, sem
  custo de licença nem de API — mantém a comparação de custo justa contra o Gemini, da seção
  da API do Gemini até a seção de conclusão.
- **Bem menor que o NLLB**: cerca de 215 milhões de parâmetros contra 600 milhões,
  confirmado carregando o modelo e somando o `numel` de cada parâmetro, não por suposição —
  um ponto de comparação também em tamanho e velocidade, não só em qualidade de tradução.
  Isso também explica por que carrega e traduz mais rápido que o NLLB.

**✅ Arquitetura e tokenização**

Assim como o NLLB, é um Transformer **encoder-decoder** — mesma arquitetura e mesma lógica
de código já detalhadas na seção do NLLB, com `MarianMTModel` no lugar de
`AutoModelForSeq2SeqLM`. As diferenças práticas ficam na tokenização:

- **Classes do `transformers`**: `MarianTokenizer` e `MarianMTModel`, não
  `AutoTokenizer`/`AutoModelForSeq2SeqLM` como no NLLB. É uma arquitetura mais antiga, da
  família Marian e OPUS-MT, com sua própria implementação no `transformers`.
- **Idioma de destino como prefixo de texto, não como parâmetro separado**: diferente do
  NLLB, que usa `forced_bos_token_id`, conforme a seção da tokenização, este modelo usa um
  token especial no **início do texto de entrada** para indicar o idioma de destino:
  `">>spa<< " + linha`. Isso mantém a indicação de idioma dentro do próprio texto, ao
  contrário do NLLB, que a mantém fora, como parâmetro do método `generate`, então é preciso
  ter cuidado para não confundir esse prefixo com conteúdo real a traduzir.

**✅ Parâmetros de geração**: usa a mesma constante `PARAMETROS_GERACAO`, definida uma única
vez na seção dos parâmetros de geração, com a mesma justificativa de determinismo e *beam
search* do NLLB: `num_beams=4`, `do_sample=False`, `early_stopping=True`, `length_penalty=1.0`,
`no_repeat_ngram_size=3`. Não são valores repetidos por coincidência entre as duas funções —
vêm da mesma constante Python, então os dois modelos não podem divergir por engano se algum
dia esses valores forem ajustados. Também carregamos em `float16` quando há GPU disponível,
pelo mesmo motivo de memória da seção dos parâmetros de geração. Como este modelo é bem menor
que o NLLB, manter os dois carregados ao mesmo tempo na GPU não chegou a ser um problema de
memória nesta placa, uma RTX 4060 — a soma dos dois, em `float16`, fica bem abaixo do que já
tínhamos ajustado para rodar só o NLLB sozinho.

</div>

In [7]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_ID_HELSINKI = "Helsinki-NLP/opus-mt-tc-big-itc-itc"
PREFIXO_DESTINO_HELSINKI = ">>spa<< "

tok_helsinki = MarianTokenizer.from_pretrained(MODEL_ID_HELSINKI)
model_helsinki = MarianMTModel.from_pretrained(
    MODEL_ID_HELSINKI,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)

def traduzir_lote_helsinki(linhas: list[str], max_length: int = 512) -> list[str]:
    linhas_com_prefixo = [PREFIXO_DESTINO_HELSINKI + l for l in linhas]
    entrada = tok_helsinki(
        linhas_com_prefixo,
        return_tensors="pt", padding=True, truncation=True, max_length=max_length
    ).to(DEVICE)
    saida = model_helsinki.generate(
        **entrada,
        max_length=max_length,
        **PARAMETROS_GERACAO,
    )
    return tok_helsinki.batch_decode(saida, skip_special_tokens=True)

/home/efraxpc/projects/llm-LectureLens/.venv/lib/python3.14/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [8]:
for arquivo in documentos_pt:
    texto_original = arquivo.read_text(encoding="utf-8")
    linhas = extrair_linhas(texto_original)

    linhas_traduzidas_helsinki = []
    for i in range(0, len(linhas), BATCH_SIZE):
        lote = linhas[i:i + BATCH_SIZE]
        linhas_traduzidas_helsinki.extend(traduzir_lote_helsinki(lote))

    texto_traduzido_helsinki = "\n".join(linhas_traduzidas_helsinki)

    arquivo_saida_helsinki = PROCESSED_DIR / f"{arquivo.stem}_espanhol_helsinki.txt"
    arquivo_saida_helsinki.write_text(texto_traduzido_helsinki, encoding="utf-8")

    print(
        f"Traduzido (Helsinki): {arquivo.name} -> {arquivo_saida_helsinki.name} "
        f"({len(linhas)} linhas, {len(texto_traduzido_helsinki)} caracteres)"
    )

Traduzido (Helsinki): Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08.vtt -> Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08_espanhol_helsinki.txt (693 linhas, 93210 caracteres)
Traduzido (Helsinki): Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10.vtt -> Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10_espanhol_helsinki.txt (809 linhas, 85031 caracteres)
Traduzido (Helsinki): Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07.vtt -> Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07_espanhol_helsinki.txt (794 linhas, 97930 caracteres)
Traduzido (Helsinki): Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05.vtt -> Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05_espanhol_helsinki.txt (843 linhas, 92945 caracteres)
Traduzido (Helsinki): Sistemas_Cognitivos_com_LLMs_aula-17-06-2026_20-05.vtt -> Sistemas_Cognitivos_com_LLMs_aula-17-06-2026_20-05_espanhol_helsinki.txt (835 linhas, 82722 caracteres)
Traduzido (Helsinki): Sistemas_Cognitivos_com_LLMs_aula-22-06-2026_20-11.vtt -> 

---

## §5 Tradução usando a API do Gemini

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Por que escolhemos a API do Gemini, especificamente o 3.1 Pro**

Até aqui usamos dois modelos baixados e rodando na nossa própria máquina, especializados só
em tradução: NLLB-200-distilled-600M e Helsinki-NLP, discutidos na seção do NLLB e na seção
do Helsinki-NLP. Para a terceira forma de traduzir, escolhemos a API do Gemini, do Google,
por motivos concretos:

- **Mesma chave já prevista pro resto do projeto**: `GEMINI_API_KEY` é a mesma chave que as
  próximas etapas do curso, de C02 a C04, também usam — reaproveitamos a infraestrutura de
  API já planejada pro projeto, em vez de introduzir um quarto provedor só pra esta
  comparação.
- **Conhecimento de mundo, não só tradução especializada**: ao contrário do NLLB e do
  Helsinki, dois modelos pequenos treinados só pra tradução, o Gemini é um modelo de
  propósito geral, com conhecimento de mundo — é exatamente esse tipo de conhecimento,
  gíria, contexto, termos técnicos, que queremos testar se faz diferença, na seção que
  compara os três modelos. Um serviço de tradução especializado, tipo Google Translate ou
  DeepL, não deixaria testar essa hipótese.
- **A versão Pro, de propósito, não Flash nem Flash-Lite**: a família Gemini tem várias
  versões, da mais rápida e barata, Flash-Lite, até a de maior qualidade, Pro. Usamos o
  **Gemini 3.1 Pro (Preview)** porque, na seção que compara os três modelos, o objetivo é medir os
  modelos locais contra o melhor resultado que a API consegue entregar, não contra a opção
  mais econômica. Se usássemos Flash-Lite e a tradução saísse pior em algum ponto, não daria
  para saber se é uma limitação de modelos remotos com conhecimento de mundo em geral ou só
  da versão mais enxuta do Gemini.
- **Custo comparável**: o Gemini 3.1 Pro (Preview) cobra **US$2,00 por milhão de
  tokens de entrada** e **US$12,00 por milhão de tokens de saída** (para prompts de até 200
  mil tokens), preços de 2026 publicados em `https://ai.google.dev/gemini-api/docs/pricing`,
  cerca de **28 vezes mais caro por token** que o Flash-Lite, US$0,10 de entrada e US$0,40 de
  saída. Traduzir uma aula inteira, com cerca de 700 linhas, deve custar, na prática, **cerca
  de US$0,25 a US$0,31** — pequeno o suficiente para
  não pesar na conclusão de custo, mais adiante na seção de conclusão.

**✅ Como isso aparece no código**

A tradução é pedida numa instrução de sistema, `system_instruction`, que diz ao modelo que ele é
um tradutor de português para espanhol e que deve manter uma linha por linha, para não perder o
alinhamento com o `.vtt` original.

**Não precisamos tokenizar o texto nós mesmos**: com o NLLB e o Helsinki, tivemos que preparar o
texto à mão para o modelo entender: cortar em pedaços, preencher com `padding` e limitar o
tamanho com `truncation` e `max_length=512`, e por isso mandávamos 16 linhas por vez, o
`BATCH_SIZE` discutido da seção da tokenização até a seção do Helsinki-NLP, um número pequeno
também por causa da memória da GPU durante o *beam search*. Com o Gemini isso desaparece:
mandamos o texto puro e quem prepara ele para o modelo é o próprio Google, do outro lado. Como
além disso o Gemini aceita textos bem mais longos, conseguimos mandar uma aula inteira, com
cerca de 700 linhas, numa única chamada, sem dividir em lotes. Um passo a menos no nosso código
também é uma fonte a menos de erro.

**✅ Parâmetros usados na chamada da API**

Assim como fizemos com os parâmetros de geração dos modelos locais, na seção dos parâmetros
de geração e na seção do Helsinki-NLP, aqui também escolhemos cada parâmetro com um motivo,
em vez de usar o que vem por padrão:

- **`temperature=0`**: a mesma ideia de determinismo do `do_sample=False` da seção dos
  parâmetros de geração — quando pedimos a mesma tradução duas vezes, queremos sempre a
  mesma resposta.
- **`seed=42`**: uma segunda tentativa de determinismo, além do `temperature=0`. Vale uma
  ressalva importante: o próprio Google diz que `seed` é melhor esforço, *best effort*, ou seja,
  ajuda bastante, mas não é uma garantia de 100%, porque o modelo roda num servidor do Google que
  pode mudar com o tempo. Isso é diferente dos modelos locais, que rodam sempre os mesmos pesos
  na nossa máquina, onde a reprodutibilidade é garantida de verdade — mais sobre isso na
  **seção das limitações observadas**.
- **`candidate_count=1`**: pedimos só uma tradução por vez, não várias opções para escolher;
  pedir mais candidatos custaria mais caro sem necessidade nenhuma aqui.
- **`thinking_config` com `types.ThinkingConfig` e `thinking_level="low"`**: diferente do Flash, o
  Gemini 3.1 Pro tem um raciocínio interno, *thinking*, ligado por padrão, e **não dá para desligar
  por completo**, só reduzir ao nível mais baixo disponível, `low`. Os tokens gastos pensando são
  cobrados como se fossem tokens de saída, e traduzir linha por linha é uma tarefa mecânica que não
  precisa de raciocínio elaborado. Usamos `thinking_level`, não o antigo `thinking_budget` (um
  parâmetro de uma versão mais velha do Gemini, que não é bem respeitado por esta versão — ver o
  problema real que isso causou na **seção das limitações observadas**).
- **`max_output_tokens=65536`**: o teto real de tokens de saída deste modelo, não um número
  arbitrário. Aumentamos esse limite depois de descobrir, na prática, que em algumas aulas o
  *thinking* consumia quase todo o orçamento antes de terminar a tradução — ver a mesma seção das
  limitações observadas para o caso concreto e como o código se protege disso.

Não usamos `top_p` nem `top_k`, outros dois parâmetros comuns de geração: com `temperature=0`, o
modelo já sempre escolhe a palavra mais provável, então esses dois parâmetros não mudariam nada
aqui — o mesmo raciocínio já usado na seção dos parâmetros de geração para explicar por que
`length_penalty=1.0` é neutro nos modelos locais.

</div>

In [9]:
%pip install google-genai -q

Note: you may need to restart the kernel to use updated packages.


In [10]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # garante a chave carregada mesmo se a célula de login (c55dabb5) não rodou antes
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client_gemini = genai.Client(api_key=GEMINI_API_KEY)

MODELO_GEMINI = "gemini-3.1-pro-preview"

SYSTEM_INSTRUCTION_GEMINI = (
    "You are a professional Portuguese-to-Spanish translator. Translate the text below line "
    "by line, keeping exactly one output line for each input line, in the same order, without "
    "adding, merging, or removing lines. Return ONLY the translated lines, nothing else."
)


def _traduzir_bloco(linhas, thinking_level=types.ThinkingLevel.LOW, max_output_tokens=65536):
    """Faz uma única chamada ao Gemini para um bloco de linhas; devolve (linhas_traduzidas,
    finish_reason) — só chama e relata, quem decide se repete é traduzir_gemini() logo abaixo."""
    texto_entrada = "\n".join(linhas)
    resposta = client_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=texto_entrada,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION_GEMINI,
            temperature=0,
            seed=42,
            candidate_count=1,
            max_output_tokens=max_output_tokens,
            # thinking_level substitui o thinking_budget antigo, que não é respeitado
            # direito pelo Gemini 3.1 Pro (Preview) — ver o caso real na seção das
            # limitações observadas.
            thinking_config=types.ThinkingConfig(thinking_level=thinking_level),
        ),
    )
    linhas_traduzidas = (resposta.text or "").strip().splitlines()
    motivo = resposta.candidates[0].finish_reason
    return linhas_traduzidas, motivo


def traduzir_gemini(linhas: list[str]) -> list[str]:
    """Traduz um bloco de linhas com o Gemini. Tenta numa única chamada; se o modelo cortar
    a resposta antes de terminar (finish_reason diferente de STOP — foi o que aconteceu de
    verdade em 3 das 8 aulas, ver a seção das limitações observadas), divide o bloco ao meio
    e tenta cada metade de novo, quantas vezes for preciso, até garantir que tudo seja
    traduzido por completo."""
    linhas_traduzidas, motivo = _traduzir_bloco(linhas)
    if motivo == types.FinishReason.STOP:
        return linhas_traduzidas

    if len(linhas) == 1:
        raise RuntimeError(
            f"Gemini não conseguiu traduzir nem uma linha só ({motivo.name}): {linhas[0]!r}"
        )

    print(
        f"  ⚠️ resposta cortada ({motivo.name}) num bloco de {len(linhas)} linhas — "
        "dividindo ao meio e tentando de novo..."
    )
    meio = len(linhas) // 2
    linhas_1 = traduzir_gemini(linhas[:meio])
    linhas_2 = traduzir_gemini(linhas[meio:])
    return linhas_1 + linhas_2

In [11]:
for arquivo in documentos_pt:
    texto_original = arquivo.read_text(encoding="utf-8")
    linhas = extrair_linhas(texto_original)

    linhas_traduzidas_gemini = traduzir_gemini(linhas)

    if len(linhas_traduzidas_gemini) != len(linhas):
        # Tratamento de erro: o Gemini deveria devolver uma linha para cada linha de entrada,
        # mas isso não é garantido — avisamos se o número não bateu, sem parar a execução.
        print(
            f"⚠️ {arquivo.name}: Gemini devolveu {len(linhas_traduzidas_gemini)} linhas, "
            f"esperávamos {len(linhas)} — pode ter juntado ou quebrado alguma linha."
        )

    texto_traduzido_gemini = "\n".join(linhas_traduzidas_gemini)
    arquivo_saida_gemini = PROCESSED_DIR / f"{arquivo.stem}_espanhol_gemini.txt"
    arquivo_saida_gemini.write_text(texto_traduzido_gemini, encoding="utf-8")

    print(
        f"Traduzido (Gemini): {arquivo.name} -> {arquivo_saida_gemini.name} "
        f"({len(linhas_traduzidas_gemini)} linhas)"
    )

Traduzido (Gemini): Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08.vtt -> Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08_espanhol_gemini.txt (693 linhas)
⚠️ Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10.vtt: Gemini devolveu 808 linhas, esperávamos 809 — pode ter juntado ou quebrado alguma linha.
Traduzido (Gemini): Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10.vtt -> Sistemas_Cognitivos_com_LLMs_aula-03-06-2026_20-10_espanhol_gemini.txt (808 linhas)
  ⚠️ resposta cortada (MAX_TOKENS) num bloco de 794 linhas — dividindo ao meio e tentando de novo...
Traduzido (Gemini): Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07.vtt -> Sistemas_Cognitivos_com_LLMs_aula-08-06-2026_20-07_espanhol_gemini.txt (794 linhas)
⚠️ Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05.vtt: Gemini devolveu 841 linhas, esperávamos 843 — pode ter juntado ou quebrado alguma linha.
Traduzido (Gemini): Sistemas_Cognitivos_com_LLMs_aula-10-06-2026_20-05.vtt -> Sistemas_Cognitivos_com_LLMs_aula-10-06

---

## §6 Comparando os três: NLLB, Helsinki-NLP e Gemini Pro

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Onde os dois modelos locais mais se parecem com o Gemini, e mesmo assim traduzem pior**

Em vez de procurar os maiores erros, aqui procuramos o oposto: os trechos onde cada modelo
local, NLLB e Helsinki, já traduziu **quase igual** ao Gemini Pro, para ver se, mesmo nesses
casos quase idênticos, o Gemini ainda faz alguma coisa melhor. Se a resposta for sim, é uma
evidência mais forte do que só olhar os erros grandes: mostra que a vantagem do Gemini aparece
em qualquer nível de comparação, não só nos casos óbvios.

**Por que comparar por palavra, e não por linha**: o Gemini às vezes junta duas linhas de
entrada em uma só resposta, o que quebra o alinhamento linha a linha com os `.vtt` originais a
partir desse ponto. Por isso comparamos **palavra por palavra, no texto inteiro**, com
`difflib`, e descartamos as diferenças triviais, só pontuação ou maiúscula e minúscula, que não
dizem nada sobre qualidade de tradução.

**Por que também olhar os trechos mais diferentes**: olhar só os trechos mais parecidos
mostra a vantagem do Gemini nos casos mais fáceis, quase idênticos entre os modelos. Mas
isso não conta a história toda — vale ver também os trechos onde a tradução do modelo
local mais se afasta da do Gemini, geralmente frases mais longas e complexas, para saber
se o problema ali é do mesmo tipo, gíria, pragmática, nome próprio, ou se aparecem erros
diferentes quando o texto fica mais difícil de traduzir. Por isso, para cada modelo
local, a comparação abaixo mostra duas tabelas: os trechos mais parecidos e os trechos
mais diferentes.

**O que encontramos nos trechos mais parecidos**: mesmo quando a diferença era de uma ou duas
palavras, o Gemini Pro corrigia coisas que os dois modelos locais erravam igual:

- **Gíria brasileira, "beleza"**: a transcrição usa "beleza" como gíria informal, próxima de
  "legal" ou "combinado", não o sentido literal de "beleza física". Os dois modelos locais
  traduzem ao pé da letra: NLLB e Helsinki dizem *"Justo es **belleza**."*, onde deveria ser
  *"Justo es **genial**."*, como traduziu o Gemini. Em outra linha da mesma conversa, o Helsinki
  chega a variar a tradução da mesma palavra dentro do próprio texto, *"Belleza, tío"* numa
  linha, *"Bien"* na linha seguinte — nenhum dos dois modelos locais reconhece que é a mesma
  gíria repetida.
- **Uso do "você" brasileiro**: a linha *"Não me respondeu. Nunca mais."* é uma reclamação **na
  segunda pessoa**, alguém dizendo "você não me respondeu", só que "você" se conjuga como se
  fosse terceira pessoa em português. NLLB e Helsinki traduzem ao pé da letra, na terceira
  pessoa: *"No me **respondió**."*, como se estivessem falando de uma terceira pessoa ausente. O
  Gemini entende que é uma fala direta e traduz corretamente na segunda pessoa: *"No me
  **respondiste**."*.

Essas duas diferenças são de **uma palavra só**, mas mudam o sentido: no primeiro caso, uma
gíria vira uma palavra errada; no segundo, uma fala direta vira uma fala sobre outra pessoa.
Nenhum dos dois modelos locais entende o contexto — traduzem palavra por palavra o que está
escrito, sem saber que "beleza" é gíria ou que "respondeu" ali se refere ao próprio
interlocutor.

**Nomes próprios: o NLLB e o Helsinki erram de formas diferentes.** Recalculamos, das cerca de
50 linhas da transcrição que começam com um rótulo `nome.sobrenome:`, como `kevin.rodrigues:`,
quantas cada modelo local preserva:

| Modelo | Linhas onde o nome se perde ou muda |
|---|---|
| NLLB-200 | cerca de um terço das linhas (~35–40%) |
| Helsinki-NLP | poucas linhas (menos de 5%) |

O NLLB erra bem mais nesse ponto específico: em várias linhas, ele simplesmente **apaga o
rótulo do nome inteiro**, por exemplo `kevin.rodrigues: Estamos desesperados.` virou só
`Estamos desesperados.`, e em um caso traduziu o nome como se fosse uma palavra comum:
`fabricio.gouveia:` virou `Fábrica.Guveia:`, como se "fabricio" fosse "fábrica", *factory*. O
Helsinki, embora erre menos vezes, também tem seus próprios problemas de grafia em nomes
próprios: `fabricio.gouveia` virou `Fabricio.gouvea`, faltou uma letra no sobrenome, e
`Guimarães` perdeu o acento e virou `Guimaraes`. O Gemini preserva os três corretamente,
`fabricio.gouveia:` e `Guimarães`.

**Erro de transcrição em termo técnico: os dois modelos locais erram igual.** A transcrição
escreveu "cuida" em vez de "CUDA" e "Open Air" em vez de "OpenAI", erros de reconhecimento de
voz do áudio original. NLLB e Helsinki copiam os dois erros ao pé da letra, *"cuida lo que es,
cuida, cuida"* e *"¿Qué dijo Open Air?"*, e mantêm a grafia inconsistente "Gpt2 Gpt3 Gpt4 Gpt5".
O Gemini reconhece os termos certos pelo contexto nos três casos: `CUDA`, `OpenAI` e `GPT-2,
GPT-3, GPT-4, GPT-5`.

| O que olhamos | NLLB-200 | Helsinki-NLP | Gemini Pro |
|---|---|---|---|
| Gíria e pragmática, "beleza", "você" | Traduz ao pé da letra, erra o sentido | Traduz ao pé da letra, erra o sentido | Entende o contexto e traduz certo |
| Nome de quem fala perdido ou alterado | Cerca de um terço das linhas | Poucas linhas | Nenhuma |
| Grafia de nomes próprios | Erra o significado, `Fábrica.Guveia` | Erra a grafia, `Guimaraes`, `Fabricio.gouvea` | Sempre correta |
| Erro de transcrição em termo técnico | Copia o erro, `cuida`, `Open Air` | Copia o erro, `cuida`, `Open Air` | Reconhece o termo certo |

**Conclusão desta comparação**: os dois modelos locais compartilham a mesma limitação de fundo:
nenhum entende gíria, pragmática ou termos técnicos fora do vocabulário comum, mas cada um
falha de um jeito diferente em nomes próprios, o NLLB erra o sentido, o Helsinki erra a grafia.
O Gemini Pro não teve nenhum desses problemas em nenhum dos casos observados, mesmo nos trechos
onde a tradução já estava quase idêntica à do modelo local. Isso reforça o que a **seção de
conclusão** vai concluir: o que faz diferença aqui é ter conhecimento de mundo, o Gemini, não
qual modelo local específico se escolhe.

</div>

In [12]:
import difflib
import string

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)

arquivo_comparacao = documentos_pt[0]
stem = arquivo_comparacao.stem

linhas_nllb = (PROCESSED_DIR / f"{stem}_espanhol.txt").read_text(encoding="utf-8").splitlines()
linhas_helsinki = (PROCESSED_DIR / f"{stem}_espanhol_helsinki.txt").read_text(encoding="utf-8").splitlines()
linhas_gemini = (PROCESSED_DIR / f"{stem}_espanhol_gemini.txt").read_text(encoding="utf-8").splitlines()

# Comparamos por palavra, não por linha (o Gemini às vezes junta linhas, quebrando o alinhamento
# 1:1): juntamos tudo num texto só e usamos o difflib para achar os trechos que mudaram.
palavras_nllb = " ".join(linhas_nllb).split()
palavras_helsinki = " ".join(linhas_helsinki).split()
palavras_gemini = " ".join(linhas_gemini).split()


def eh_diferenca_trivial(a: list[str], b: list[str]) -> bool:
    """Descarta diffs de 1 palavra que só mudam pontuação/maiúscula (não interessam aqui)."""
    return len(a) == 1 and len(b) == 1 and a[0].strip(string.punctuation).lower() == b[0].strip(string.punctuation).lower()


def mostrar_tabela(df):
    """Mostra um DataFrame com o texto das células por extenso, quebrando linha em vez de
    truncar — mesma linguagem visual (azul #eef6ff) do restante do projeto (ver c03)."""
    display(df.style.set_properties(**{
        "text-align": "left", "white-space": "pre-wrap", "vertical-align": "top", "max-width": "420px",
    }).set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("background-color", "#eef6ff")]},
    ]))


def montar_tabela_trechos(palavras_local, palavras_gem, trechos, contexto=6):
    """Monta uma tabela com o contexto ao redor de cada trecho que mudou, o próprio trecho
    marcado entre »...«, e o tamanho da diferença em palavras."""
    linhas = []
    for tag, i1, i2, j1, j2 in trechos:
        c_a1, c_a2 = max(0, i1 - contexto), i2 + contexto
        c_b1, c_b2 = max(0, j1 - contexto), j2 + contexto
        linhas.append({
            "modelo local": f"...{' '.join(palavras_local[c_a1:i1])} »{' '.join(palavras_local[i1:i2])}« {' '.join(palavras_local[i2:c_a2])}...",
            "Gemini Pro": f"...{' '.join(palavras_gem[c_b1:j1])} »{' '.join(palavras_gem[j1:j2])}« {' '.join(palavras_gem[j2:c_b2])}...",
            "palavras diferentes": max(i2 - i1, j2 - j1),
        })
    return pd.DataFrame(linhas)


def comparar_com_gemini(palavras_local: list[str], palavras_gem: list[str], nome_local: str, n: int = 5, contexto: int = 6):
    matcher = difflib.SequenceMatcher(None, palavras_local, palavras_gem, autojunk=False)
    trechos = [op for op in matcher.get_opcodes() if op[0] == "replace"]
    trechos = [
        (tag, i1, i2, j1, j2) for tag, i1, i2, j1, j2 in trechos
        if not eh_diferenca_trivial(palavras_local[i1:i2], palavras_gem[j1:j2])
    ]

    # Ordena do trecho MAIS SIMILAR (menor diferença) para o MAIS DIFERENTE (maior diferença) —
    # olhamos as duas pontas: onde o modelo local já quase acerta, e onde mais se afasta do
    # Gemini, geralmente em frases mais longas e complexas.
    trechos.sort(key=lambda op: max(op[2] - op[1], op[4] - op[3]))

    print(f"\n=== {nome_local} vs. Gemini Pro — os {n} trechos MAIS PARECIDOS entre si ===")
    mostrar_tabela(montar_tabela_trechos(palavras_local, palavras_gem, trechos[:n], contexto))

    print(f"\n=== {nome_local} vs. Gemini Pro — os {n} trechos MAIS DIFERENTES (mais complexos) ===")
    mostrar_tabela(montar_tabela_trechos(palavras_local, palavras_gem, list(reversed(trechos))[:n], contexto))


print(f"Comparando: {stem}")
comparar_com_gemini(palavras_nllb, palavras_gemini, "NLLB")
comparar_com_gemini(palavras_helsinki, palavras_gemini, "Helsinki")

Comparando: Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08

=== NLLB vs. Gemini Pro — os 5 trechos MAIS PARECIDOS entre si ===


,modelo local,Gemini Pro,palavras diferentes
0,"...Ferreira: ¿Qué pasó? Kevin. Rodrigues: Ah, »¿quieres?« Entrega, ¿no? Hoy es el día...","...Guimarães Ferreira: ¿Qué pasó? kevin.rodrigues: Ah, »¿quiere?« Entrega, ¿no? Hoy es día de...",1
1,...el trabajo fue realmente bien hecho. »Fábrica.Guveia:« Alivio. Lo tendré después de que...,...el trabajo realmente estuvo bien hecho. »fabricio.gouveia:« Alivio. Lo tendré después de recibir...,1
2,"...Bernardo. Otra vez. Fernando Guimarães Ferreira: »Está« bien, Jorge, habla conmigo. Jorge Martins...","...Jorge Bernardo. Nuevamente. Fernando Guimarães Ferreira: »Todo« bien, Jorge, dime. Jorge Martins Bernardo:...",1
3,"...belleza. ¿Te has olvidado de mí, »verdad?« Fernando Guimarães Ferreira: ¿Por qué? Jorge...","...Bernardo: Belleza. Te olvidaste de mí, »¿verdad?« Fernando Guimarães Ferreira: ¿Por qué? Jorge...",1
4,...qué? Jorge Martins Bernardo: No me »respondió.« Nunca más. Jorge Martins Bernardo: Los...,...qué? Jorge Martins Bernardo: No me »respondiste.« Nunca más. Allá en. Jorge Martins...,1



=== NLLB vs. Gemini Pro — os 5 trechos MAIS DIFERENTES (mais complexos) ===


,modelo local,Gemini Pro,palavras diferentes
0,...ahora en la disciplina anterior de »sacar este top Worlds.« Fernando Guimarães Ferreira: Por ejemplo. Así...,"...ahora en la disciplina anterior de »quitar las stop words. Mirar el contexto, hacer eso, ese trabajo de especialistas entre comillas, ese machine learning de especialistas, lo delegamos al Deep Learning, porque en algún momento, va aprendiendo en las capas que esa palabra no es importante para el contexto,« Fernando Guimarães Ferreira: por ejemplo. Entonces...",43
1,...tenemos cosas muy poderosas y muy »rápidas que se están produciendo.« Fernando Guimarães Ferreira: por un modelo...,"...tenemos cosas muy poderosas y muy »rápidas, siendo producidas. Esta figura de dos mil catorce es la primera figura generada por imagen, por una imagen generada completamente artificial,« Fernando Guimarães Ferreira: por un modelo...",22
2,"...primero, el padre de la inteligencia »artificial.« Fernando Guimarães Ferreira: el ejemplo de...","...primero, el padre de la inteligencia »artificial se ríe uno. Ya hablamos de él en la otra disciplina con la cuestión de la prueba de Turing. En fin,« Fernando Guimarães Ferreira: el ejemplo de...",22
3,"...estar consumiendo modelos, pero no está »operationalizando, ¿verdad?« Fernando Guimarães Ferreira: de modelos a...","...estar consumiendo modelos, pero no está »operacionalizando, no está. Hasta un ingeniero de un ingeniero de machine Learning, que es un ingeniero capaz de viabilizar el uso« Fernando Guimarães Ferreira: de modelos a...",21
4,...de Río tiene una asociación con »Co Pilot.« Fernando Guimarães Ferreira: Entonces terminamos. Es...,...Público de Río tiene asociación con »Copilot. El Ministerio Público de São Paulo tiene asociación con OpenAI dónde encaja eso en el proceso legal penal.« Fernando Guimarães Ferreira: Entonces terminamos. Es...,19



=== Helsinki vs. Gemini Pro — os 5 trechos MAIS PARECIDOS entre si ===


,modelo local,Gemini Pro,palavras diferentes
0,...Profesor. Fernando Guimarães Ferreira: ¿Por qué »desesperadas?« Fernando Guimarães Ferreira: ¿Qué pasó? kevin.rodrigues:...,...profesor. Fernando Guimarães Ferreira: ¿Por qué »desesperados?« Fernando Guimarães Ferreira: ¿Qué pasó? kevin.rodrigues:...,1
1,"...Guimarães Ferreira: ¿Qué pasó? kevin.rodrigues: Ah, »¿quieres?« Entrega, ¿verdad? Hoy es día de...","...Guimarães Ferreira: ¿Qué pasó? kevin.rodrigues: Ah, »¿quiere?« Entrega, ¿no? Hoy es día de...",1
2,"...¿Qué pasó? kevin.rodrigues: Ah, ¿quieres? Entrega, »¿verdad?« Hoy es día de entrega. Entonces...","...¿Qué pasó? kevin.rodrigues: Ah, ¿quiere? Entrega, »¿no?« Hoy es día de entrega. Así...",1
3,...trabajo realmente ha sido bien hecho. »Fabricio.gouvea:« Alivio. Voy a tener después de...,...el trabajo realmente estuvo bien hecho. »fabricio.gouveia:« Alivio. Lo tendré después de recibir...,1
4,...esperando. Corrección: ahí ya está esperando. »Correción.« Fernando Guimarães Ferreira: Justo. Justo es...,...esperando. Corrección: ahí ya está esperando. »Corrección.« Fernando Guimarães Ferreira: Justo. Justo es...,1



=== Helsinki vs. Gemini Pro — os 5 trechos MAIS DIFERENTES (mais complexos) ===


,modelo local,Gemini Pro,palavras diferentes
0,"...y habla de artículos de inteligencia »comput« Fernando Guimarães Ferreira: van siendo consolidadas,...","...y habla en artículos de inteligencia »computacional. Hablo de inteligencia artificial. Entonces hay un momento que esas definiciones, ellas« Fernando Guimarães Ferreira: van siendo consolidadas,...",13
1,"...de habla, pero es un interés, »¿verdad?« Fernando Guimarães Ferreira: está trabajando con...","...de habla, pero es un interés, »¿no? Hasta ¿no? Soy hijo de artista que« Fernando Guimarães Ferreira: está trabajando con...",8
2,...de cuál sea el profesional que »lo haga?« Fernando Guimarães Ferreira: sometida a una...,...de ¿cuál es el profesional que »hace eso? Calidad de datos está muy« Fernando Guimarães Ferreira: subyugada a una...,7
3,"...Efrain Colmenares: Así que eso es »todo. Quisiera« que nos dijeras, por ejemplo, la...","...estadísticos y tal Efrain Colmenares: es »Efrain Colmenares: entonces, es eso. Quería ver« que nos hable, por ejemplo, la...",7
4,"...esta aquí. Donde. Jorge Martins Bernardo: »Eso es, tío. Eso es.« Y ahí, lo que los chicos...","...aquí. Dónde está. Jorge Martins Bernardo: »Es eso ahí, hombre. Es eso ahí.« Y ahí, lo que los tipos...",7


---

## §7 Limitações observadas

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Limitações observadas — NLLB-200-distilled-600M**

- **Apaga ou erra o nome de quem fala com frequência**: como vimos em detalhe na seção que
  compara os três modelos, das cerca de 50 linhas da transcrição que começam com um nome de
  pessoa, o NLLB perdeu ou alterou esse nome em cerca de um terço delas, a pior taxa entre os
  três modelos testados, e em um caso chegou a traduzir o nome `fabricio.gouveia` como se
  fosse a palavra comum "fábrica", `Fábrica.Guveia`.
- **Gíria, pragmática e termo técnico não são corrigidos**: também detalhado na seção que
  compara os três modelos — "beleza", gíria, virou "belleza", sentido literal errado;
  "respondeu", fala direta em "você", virou "respondió", como se falasse de outra pessoa; e
  "cuida" e "Open Air" não viraram "CUDA" e "OpenAI".
- **Perda de contexto entre lotes**: cada lote de linhas, de tamanho `BATCH_SIZE=16`, é
  traduzido de forma independente dos demais, sem memória do que veio antes. Isso pode gerar
  inconsistências de terminologia ou perda de coerência em diálogos longos, algo que um modelo
  com janela de contexto completa não teria.
- **Teto de qualidade de um modelo pequeno e destilado**: mesmo sendo uma arquitetura mais
  recente e treinada para 200 idiomas, `nllb-200-distilled-600M` continua sendo um modelo
  pequeno e especializado, 600 milhões de parâmetros, muito abaixo de um modelo fundacional
  moderno.
- **Uso de memória da GPU durante o *beam search***: com `num_beams=4`, a memória necessária
  cresce junto com o tamanho do lote. Um lote grande demais, como `BATCH_SIZE=32`, pode
  estourar a memória da placa de vídeo, erro `CUDA error: out of memory`. Por isso carregamos o
  modelo em `float16` e usamos `BATCH_SIZE=16`, discutido na seção dos parâmetros de geração,
  valores que já testamos e funcionam nesta GPU, uma RTX 4060.
- **Custo computacional local**: traduzir uma aula com cerca de 700 linhas do transcript levou
  cerca de 2 minutos numa GPU NVIDIA RTX 4060, viável neste ambiente, mas ainda assim depende
  de ter uma placa de vídeo disponível, ver a **seção de conclusão**.

**✅ Limitações observadas — Helsinki-NLP, `opus-mt-tc-big-itc-itc`**

- **Erra a grafia de nomes próprios**: como vimos na seção que compara os três modelos,
  embora perca o nome de quem fala bem menos vezes que o NLLB, poucas linhas, menos de 5%,
  quando erra costuma ser um problema de **grafia**, não de significado: `fabricio.gouveia`
  virou `Fabricio.gouvea`, faltou uma letra, e `Guimarães` perdeu o acento e virou
  `Guimaraes`.
- **Cobertura de idiomas mais estreita**: cobre uma família de idiomas itálicos, `itc-itc`:
  português, espanhol, italiano, francês, entre outros, contra 200 idiomas de famílias bem
  diferentes no NLLB. É suficiente para português-espanhol, mas bem menos versátil se o
  projeto precisar traduzir para outro idioma no futuro.
- **Mesma falta de conhecimento de mundo que o NLLB**: como vimos na seção que compara os
  três modelos, também não reconhece gíria, "beleza", pragmática, "você" na segunda pessoa,
  nem termos técnicos com erro de transcrição, CUDA, OpenAI. A limitação de fundo é a mesma,
  só muda onde cada modelo erra mais.
- **Depende do prefixo de texto correto**: o idioma de destino é indicado por um token no
  início do próprio texto de entrada, `>>spa<< `, discutido na seção do Helsinki-NLP, não por
  um parâmetro separado como no NLLB. Um erro ao montar esse prefixo, como esquecê-lo, faz o
  modelo escolher o idioma de saída sozinho, sem aviso de erro.
- **Custo computacional local menor que o NLLB**: por ser um modelo bem menor, cerca de 215
  milhões de parâmetros contra 600 milhões, traduzir a mesma aula, com cerca de 700 linhas,
  levou cerca de 1 minuto nesta GPU, mais rápido que o NLLB, mas ainda assim depende de placa
  de vídeo disponível, ver a **seção de conclusão**.

**✅ Limitações observadas — Gemini Pro**

- **Precisa de internet e de uma chave de API**: sem `GEMINI_API_KEY` configurada e sem
  conexão, a tradução simplesmente não roda, diferente dos modelos locais, que, uma vez
  baixados, funcionam offline.
- **Cada tradução tem um custo bem maior que o Flash-Lite**: cerca de US$0,25 a US$0,31 por
  aula, discutido na seção da API do Gemini, ainda pequeno em termos absolutos, mas cerca de
  28 vezes mais caro por token do que a versão mais econômica da família Gemini.
- **O texto sai da nossa máquina**: para traduzir, mandamos o conteúdo da aula para os
  servidores do Google. Para este projeto de curso isso não é um problema, mas é algo a
  considerar se o conteúdo fosse sigiloso.
- **A reprodutibilidade não é 100% garantida**: fixamos `temperature=0` e `seed=42`,
  discutido na seção da API do Gemini, para tentar sempre a mesma resposta, mas o próprio
  Google avisa que `seed` é melhor esforço, não uma garantia; o modelo roda num servidor que
  pode mudar com o tempo. Isso é diferente dos modelos locais, que sempre rodam os mesmos
  pesos na nossa máquina, com reprodutibilidade garantida de verdade.
- **O *thinking* não pode ser totalmente desligado**: diferente do Flash, o Gemini 3.1 Pro
  sempre gasta ao menos alguns tokens pensando antes de responder, discutido na seção da API
  do Gemini, pouco na prática na maioria das aulas, mas é um custo que não existe nos modelos
  locais nem no Flash-Lite.
- **Em alguns casos, o texto traduzido parava no meio de uma frase, sem terminar a aula inteira (Foi Corrigido programaticamente)**: A causa:
  `thinking_budget`, um parâmetro antigo, não é bem respeitado pelo Gemini 3.1 Pro (Preview),
  então o modelo "pensava" bem mais do que o esperado nas aulas com fala mais desorganizada, e
  esse pensamento consumia o limite de tokens antes de terminar a tradução.

  A correção tem duas partes, e a segunda é a que resolve o problema de verdade. Primeiro,
  trocamos `thinking_budget` por `thinking_level="low"`, o parâmetro certo para essa versão
  do Gemini, e aumentamos `max_output_tokens` para 65536, o teto real que o modelo aceita —
  isso já reduz bastante o risco, mas não é uma garantia, porque a própria documentação do
  Google descreve `thinking_level` como uma "alocação relativa" de raciocínio, não um limite
  garantido. Por isso o código passou a conferir também o `finish_reason` de cada resposta: se
  o Gemini não terminar (`finish_reason` diferente de `STOP`), o código divide o texto ao meio
  e tenta de novo em cada metade, separadamente. Isso garante mesmo, e não só na prática,
  porque cada nova tentativa trabalha com metade das linhas da tentativa anterior — o bloco
  sempre acaba pequeno o suficiente para caber no limite de tokens, não importa quanto o
  modelo precise pensar. É essa segunda parte que transforma a correção num número maior e
  torcendo para dar certo em uma garantia de que a aula inteira sempre será traduzida por
  completo.
- **É uma versão Preview, não estável**: `gemini-3.1-pro-preview` pode ser alterado ou
  descontinuado pelo Google com menos aviso prévio do que uma versão estável (GA), diferente
  dos modelos locais, cujos pesos ficam fixos na nossa máquina para sempre. Vale revisitar
  esta escolha quando uma versão estável equivalente, `gemini-3.1-pro`, for lançada.

</div>

---

## §8 O que encaixa melhor no que precisamos

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**✅ Comparando as três opções com o que o projeto realmente precisa**

**O que precisamos**: traduzir as aulas de português para espanhol, sempre do mesmo jeito,
determinístico, sem gastar muito, e deixando os arquivos prontos para as próximas etapas do
curso, a partir de C02, tanto em espanhol quanto em português, linha por linha.

Isso pede, na prática:

1. Uma tradução boa o suficiente entre duas línguas parecidas, sem precisar raciocinar sobre
   o texto.
2. Sempre dar a mesma resposta para a mesma entrada.
3. Custo baixo e previsível, mesmo rodando o notebook várias vezes.
4. Não depender de coisas que podem falhar, como internet, servidor ou chave de terceiros,
   mais do que o necessário.
5. Rodar num tempo razoável.
6. Deixar o texto, em espanhol e português, pronto para ser usado depois, sem precisar ainda
   cortar em pedaços nem gerar embeddings; isso fica para outra etapa do curso, C05, RAG.

**Como cada opção se sai nesses pontos**: o NLLB-200, discutido da seção do NLLB até a seção
dos parâmetros de geração, e o Helsinki-NLP, na seção do Helsinki-NLP, são bons no ponto 2,
sempre dão a mesma resposta, e razoáveis no ponto 1 — mas, como vimos na seção que compara os
três modelos, os dois ainda erram gíria, pragmática e termos técnicos com problema de
transcrição, e cada um erra nomes próprios do seu jeito: o NLLB em cerca de um terço das
linhas com nome, o Helsinki em poucas linhas, mas com erros de grafia. Além disso, os dois
dependem de ter uma placa de vídeo disponível de graça, ponto 3, e de configurar tokenização
manualmente. O Gemini Pro, discutido na seção da API do Gemini, também acerta os pontos 1 e
2, com `temperature=0` e `seed=42`, ainda que sem a garantia de 100% de um modelo local, ver
a seção das limitações observadas, e vai melhor no ponto 1 na prática, como mostra a seção
que compara os três modelos, do que os dois modelos locais, além de ter uma vantagem a mais:
não precisamos preparar o texto para ele, tokenizar, cortar em pedaços; mandamos o texto puro
e ele cuida disso sozinho, o que deixa o código mais simples e com menos chance de erro.

No ponto 3, custo, é onde a diferença fica mais clara: um modelo local só é gratuito se já
tivermos uma placa de vídeo disponível; num servidor de verdade, isso custa dinheiro de
verdade, cerca de US$384 por mês numa AWS `g4dn.xlarge`, e vale tanto para o NLLB quanto para
o Helsinki. O Gemini Pro não precisa de servidor nenhum e custa **cerca de US$0,25 a US$0,31
por aula**, mais caro por tradução do que a versão Flash-Lite, cerca de US$0,01, mas ainda
muito abaixo do custo de um servidor com GPU.

**O que perdemos e o que ganhamos**: como vimos na seção das limitações observadas, cada
opção tem seus problemas — o NLLB apaga ou erra nomes com frequência e precisa de GPU; o
Helsinki erra a grafia de nomes e cobre menos idiomas; o Gemini Pro precisa de internet e
custa mais por tradução do que a versão mais barata da família. Olhando os pontos 1 a 6
juntos, o Gemini Pro sai na frente em quase tudo: traduz melhor que os dois modelos locais,
como mostra a seção que compara os três modelos, é mais simples de programar, como mostra a
seção da API do Gemini, e continua muito mais barato do que manter um servidor com GPU, como
já mostrado acima. O único ponto onde um modelo local ganha é a reprodutibilidade de 100%
garantida, ponto 2, mas isso só importa se a pequena diferença de determinismo do Gemini,
discutida na seção da API do Gemini e na seção das limitações observadas, for um problema
real para o caso de uso, o que não é o caso aqui.

**Conclusão**: entre as três opções testadas neste notebook, o **Gemini Pro é a escolha
recomendada** para traduzir as aulas, pela qualidade — gíria, pragmática, nomes e termos
técnicos mais corretos do que qualquer um dos dois modelos locais — e por ainda custar
muito menos do que manter um servidor com GPU, mesmo sendo a versão mais cara da família
Gemini. Um modelo local continua sendo uma alternativa válida apenas quando já existe uma
placa de vídeo disponível sem custo extra, como no ambiente deste curso.

</div>

---

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**📋 Resumo dos pontos demonstrados**

1. Primeiro modelo local, NLLB-200-distilled-600M: por que escolhemos esse modelo e por que
   é encoder-decoder, nem encoder-only nem decoder-only, na seção
   **"Primeiro modelo local: NLLB-200-distilled-600M"**.
2. Uso de tokenização no NLLB, na seção **"Justificativa da tokenização"**.
3. Parâmetros de geração no NLLB, na seção **"Parâmetros de geração"**.
4. Segundo modelo local, Helsinki-NLP: por que escolhemos esse modelo, e diferenças de
   arquitetura e mecanismo de idioma de destino frente ao NLLB, na seção
   **"Segundo modelo local: Helsinki-NLP"**.
5. Terceira forma de traduzir: por que escolhemos a API do Gemini, especificamente o 3.1
   Pro, com seus próprios parâmetros justificados, incluindo o controle do *thinking*, e sem
   precisar tokenizar nada à mão, na seção **"Tradução usando a API do Gemini"**.
6. Comparação entre NLLB-200, Helsinki-NLP e Gemini Pro nos trechos mais parecidos entre si,
   com exemplos reais de gíria, pragmática, nomes próprios e termos técnicos que só o Gemini
   traduziu corretamente, na seção
   **"Comparando os três: NLLB, Helsinki-NLP e Gemini Pro"**.
7. Limitações observadas nas três opções, na seção **"Limitações observadas"**.
8. Qual opção encaixa melhor no que o projeto precisa — incluindo custo de infraestrutura
   contra custo por uso, e por que nenhum dos dois modelos locais resolve o problema de
   qualidade — e conclusão final, na seção **"O que encaixa melhor no que precisamos"**.
9. Qual tarefa de NLP este notebook demonstrou, tradução automática, por que essa é a tarefa
   certa pro propósito do projeto — ajudar estudantes hispanofalantes em países lusófonos a
   entender aulas em português, incluindo gírias — e o que vem a seguir em C02, sumarização
   por tópico e Question Answering (Q&A).

</div>

---

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**🎯 Tarefa de NLP demonstrada neste notebook**

A tarefa de NLP demonstrada neste notebook foi **tradução automática**, de português para
espanhol — uma tarefa de geração de sequência a partir de outra sequência (*seq2seq*), onde a
entrada e a saída são textos completos, não uma classificação nem uma pontuação.

O propósito deste projeto é servir de solução para estudantes
hispanofalantes que moram em países lusófonos e precisam entender, em espanhol, aulas dadas
em português — incluindo gírias e expressões informais, que uma tradução literal não resolve,
como já mostrou o exemplo real da seção que compara os três modelos ("beleza" traduzido
corretamente como "genial", não "belleza"). Por isso a primeira etapa do pipeline precisa
converter o conteúdo original das aulas para espanhol, com fidelidade e de forma reprodutível,
antes de qualquer outra tarefa poder usar esse corpus bilíngue — e as próximas etapas, a
partir de C02, comparam técnicas de NLP nos dois idiomas, português e espanhol. Tradução
também é o exemplo mais direto de por que um modelo encoder-decoder, nem encoder-only nem
decoder-only, discutido na seção do NLLB e na seção do Helsinki-NLP, é necessário aqui: a
tarefa exige entender a entrada inteira antes de gerar uma sequência nova.

**O que vem a seguir, em C02**: com o corpus já traduzido, C02 demonstra duas outras tarefas
de NLP sobre esse mesmo material, **sumarização por tópico** e **Question Answering (Q&A)**,
usando arquiteturas diferentes das usadas aqui.

</div>

---

## §9 Conclusão

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**📋 O que a comparação dos três tradutores mostrou, e o que isso decide para o corpus**

O caminho deste notebook foi: limpar as transcrições WEBVTT mantendo uma linha por cue,
traduzir a mesma aula de português para espanhol por três caminhos — dois modelos locais,
`nllb-200-distilled-600M` e `Helsinki-NLP/opus-mt-tc-big-itc-itc`, e a API do Gemini 3.1
Pro — e comparar qualidade e custo palavra por palavra. Esta conclusão junta o que essa
comparação mostrou, em três perguntas: onde a tradução funcionou, onde os modelos locais
falharam, e o que isso decide para o corpus que segue para as próximas etapas.

**Onde a tradução funcionou.** Português e espanhol são línguas próximas, e nos trechos
mais simples os três caminhos acertam a maior parte do texto. Os dois modelos locais têm
duas vantagens reais: são **determinísticos**, sempre dão a mesma tradução para a mesma
entrada, e rodam **de graça** numa GPU já disponível. Mas a comparação palavra por palavra
mostrou que, mesmo nos trechos quase idênticos, o Gemini Pro ainda corrigia o que os locais
erravam — sobretudo em gíria, onde a tradução literal falha.

<div style="background:#eafaf0;border-left:4px solid #2f9e5c;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
✅ <strong>Exemplo:</strong> a gíria <code>"beleza"</code> (no sentido de "combinado",
"legal") foi traduzida pelo Gemini como <em>"genial"</em>, enquanto o NLLB e o Helsinki a
verteram ao pé da letra como <em>"belleza"</em> — o sentido literal errado.
</div>

**Onde os modelos locais falharam.** O problema mais sério apareceu nos **nomes próprios**.
Das cerca de 50 linhas da transcrição que começam com o nome de quem fala, o NLLB perdeu ou
alterou o nome em cerca de **um terço** delas — a pior taxa entre os três. O Helsinki erra
bem menos, **menos de 5%**, mas quando erra é por **grafia**, não por significado. Além
disso, nenhum dos dois corrige gíria, pragmática do "você" ou termos técnicos mal
transcritos (como "cuida"/"Open Air" em vez de "CUDA"/"OpenAI"), e ambos traduzem cada lote
de linhas sem memória do anterior, o que pode gerar inconsistências em diálogos longos.

<div style="background:#fff8e6;border-left:4px solid #d9a404;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
⚠️ <strong>Exemplo:</strong> o NLLB traduziu o nome <code>fabricio.gouveia</code> como se
fosse a palavra comum <em>"fábrica"</em>, virando <code>Fábrica.Guveia</code> — um nome
próprio transformado em substantivo.
</div>

**Como isso decide o corpus e o custo.** Há um trade-off honesto. O NLLB é o modelo cuja
tradução alimenta as próximas etapas do pipeline, por ser determinístico e gratuito na GPU
já disponível; mas a comparação deixa claro que o **Gemini Pro vai à frente em qualidade**,
justamente nos pontos que mais importam para o propósito do projeto — gíria, pragmática e
nomes próprios. Do lado do custo, a conta depende do volume: um servidor com GPU próprio
custa cerca de **US$384 por mês** (uma instância AWS `g4dn.xlarge`), fixo, independente do
uso, enquanto a API cobra só pelos tokens efetivamente traduzidos; e traduzir uma aula de
cerca de 700 linhas levou cerca de **2 minutos** numa RTX 4060. E quanto custa, em dinheiro,
**uma tradução** na API? A conta é dominada pela saída: o Gemini Pro cobra US$2,00 por milhão
de tokens de entrada e US$12,00 por milhão de saída (os preços da seção da API do Gemini), e
uma tradução devolve uma saída quase do tamanho da entrada — a aula inteira reescrita em
espanhol —, mais o pouco de *thinking* que o Pro não deixa desligar. Para uma aula de cerca
de 700 linhas, isso dá **cerca de US$0,25 a US$0,31 por tradução**, a estimativa já usada
nas seções da API do Gemini e da comparação de custos. E este corpus tem **8 aulas**, todas
de fato traduzidas com o Pro — contando só esse gasto (a GPU local dos dois modelos de
comparação já estava disponível sem custo extra, então não entra dinheiro novo na conta), o
corpus completo custou:

<div style="background:#eafaf0;border-left:4px solid #2f9e5c;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
💰 <strong>Custo estimado do corpus completo: 8 aulas × US$0,25–0,31 ≈ US$2,00 a
US$2,48</strong> (na prática, ~US$2,50) — pago <strong>uma única vez</strong>. É todo o
custo de preparação do corpus bilíngue que as próximas etapas, de C02 ao RAG de C05,
reutilizam sem pagar de novo.
</div>

A lição é que nenhum modelo
local pequeno, por mais recente que seja, fecha a diferença de qualidade — a escolha entre
o modelo local e a API é uma decisão de qualidade contra custo, não de conveniência.

**Implicações para as próximas etapas.** Ao final, o corpus bilíngue fica pronto: a tradução
em espanhol e o português já limpo, linha por linha, em `data/processed/`. É esse par que as
próximas etapas usam — a partir de C02, que compara técnicas de prompting nos dois idiomas, e
mais adiante os embeddings e o RAG em C05. E a própria tarefa deste notebook, tradução
automática, é o exemplo mais direto de por que uma arquitetura **encoder-decoder** (nem
encoder-only, nem decoder-only) é necessária aqui: é preciso entender a frase inteira de
entrada antes de gerar a sequência de saída.

</div>